### IVOIRE Project — Order Volume Forecasting by State
- Weekly revenue forecast (Prophet) for the top 15 states by order volume.
- One model per state, results consolidated into a single `fct_volume_forecast`
table linked by `state`.

In [41]:
import pandas as pd
from sqlalchemy import create_engine
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

### 1. Connection & extraction
- Query already returns a continuous (state x date) grid with zero-filled gaps
required so Prophet doesn't treat missing weeks as missing data.

In [ ]:
engine = create_engine(
    "postgresql+psycopg2://postgres:@localhost:5432/Brazilian E-Commerce [IVOIRE]"
)

query = """
WITH top_states AS (
    SELECT state
    FROM fct_logistics_kpis_state
    ORDER BY revenue DESC
    LIMIT 15
),

date_state_grid AS (
    SELECT d.full_date AS ds, ts.state AS state
    FROM dim_dates d
    CROSS JOIN top_states ts
    WHERE d.full_date BETWEEN '2016-09-04' AND '2018-09-03'
),

actual_volume AS (
    SELECT
        c.customer_state AS state,
        d.full_date AS ds,
        SUM(f.total_line_cost) AS revenue
    FROM fact_shipments f
    INNER JOIN dim_customers c ON f.customer_id = c.customer_id
    INNER JOIN dim_dates d ON f.purchase_date_key = d.date_key
    WHERE c.customer_state IN (SELECT customer_state FROM top_states)
    GROUP BY c.customer_state, d.full_date
)

SELECT
    g.state AS state,
    g.ds,
    COALESCE(av.revenue, 0) AS y
FROM date_state_grid g
LEFT JOIN actual_volume av
    ON g.state = av.state
    AND g.ds = av.ds
ORDER BY g.state, g.ds ASC;
"""

df_daily = pd.read_sql(query, engine)
df_daily["ds"] = pd.to_datetime(df_daily["ds"])

### 2. Resample to weekly
- Weekly granularity gives Prophet enough data points (~100 weeks over ~2
years) to learn a seasonal pattern, unlike monthly (~24 points, too short).

In [43]:
df_weekly = (
    df_daily
    .set_index("ds")
    .groupby("state")
    .resample("W")["y"]
    .sum()
    .reset_index()
)


In [44]:
top_states = df_weekly['state'].unique()

### 3. Per-state Prophet loop
- Train/test split (last 8 weeks held out) validates each model before
trusting its forward-looking forecast. MAE/MAPE reported per state.

In [45]:
df_weekly.head(500)

,state,ds,y
0,BA,2016-09-04,0.00
1,BA,2016-09-11,0.00
2,BA,2016-09-18,0.00
3,BA,2016-09-25,0.00
4,BA,2016-10-02,0.00
...,...,...,...
495,GO,2018-01-14,4922.23
496,GO,2018-01-21,4184.01
497,GO,2018-01-28,4405.71
498,GO,2018-02-04,6265.53


In [46]:
HOLDOUT_WEEKS = 8
FORECAST_HORIZON_WEEKS = 12

In [47]:
all_forecasts = []
validation_results = []

for state in top_states:
    state_df = df_weekly[df_weekly['state'] == state][['ds', 'y']].reset_index(drop=True)

    train = state_df.iloc[:-HOLDOUT_WEEKS]
    test = state_df.iloc[-HOLDOUT_WEEKS:]

    model = Prophet(weekly_seasonality=False, yearly_seasonality=True)
    model.fit(train)

    # Validation: predict over the held-out period, compare to actuals

    future_test = model.make_future_dataframe(periods=HOLDOUT_WEEKS, freq='W')
    forecast_test = model.predict(future_test)
    predicted = forecast_test.tail(HOLDOUT_WEEKS)['yhat'].values
    actual = test['y'].values

    mae = mean_absolute_error(actual, predicted)
    mape = mean_absolute_percentage_error(actual, predicted)
    validation_results.append({"state": state, "mae": mae, "mape": mape})

    # Final model: retrained on the FULL series, forecasting genuinely ahead

    final_model = Prophet(weekly_seasonality=False, yearly_seasonality=True)
    final_model.fit(state_df)
 
    future = final_model.make_future_dataframe(periods=FORECAST_HORIZON_WEEKS, freq="W")
    forecast = final_model.predict(future)

    forecast["state"] = state
    all_forecasts.append(forecast[["state", "ds", "yhat", "yhat_lower", "yhat_upper"]])

13:11:53 - cmdstanpy - INFO - Chain [1] start processing
13:11:53 - cmdstanpy - INFO - Chain [1] done processing
13:11:53 - cmdstanpy - INFO - Chain [1] start processing
13:11:53 - cmdstanpy - INFO - Chain [1] done processing
13:11:53 - cmdstanpy - INFO - Chain [1] start processing
13:11:54 - cmdstanpy - INFO - Chain [1] done processing
13:11:54 - cmdstanpy - INFO - Chain [1] start processing
13:11:54 - cmdstanpy - INFO - Chain [1] done processing
13:11:54 - cmdstanpy - INFO - Chain [1] start processing
13:11:54 - cmdstanpy - INFO - Chain [1] done processing
13:11:54 - cmdstanpy - INFO - Chain [1] start processing
13:11:54 - cmdstanpy - INFO - Chain [1] done processing
13:11:54 - cmdstanpy - INFO - Chain [1] start processing
13:11:54 - cmdstanpy - INFO - Chain [1] done processing
13:11:54 - cmdstanpy - INFO - Chain [1] start processing
13:11:54 - cmdstanpy - INFO - Chain [1] done processing
13:11:55 - cmdstanpy - INFO - Chain [1] start processing
13:11:55 - cmdstanpy - INFO - Chain [1]

### 4. Validation summary
- Check MAPE per state before trusting the forecast — a high MAPE means the
model struggled to fit that state's historical pattern.

In [48]:
validation_df = pd.DataFrame(validation_results).sort_values("mape")
print(validation_df)

   state           mae          mape
14    SP  45157.318362  9.929203e+01
7     MT   2703.476481  4.460224e+18
5     MA   2577.646386  5.313252e+18
13    SC   5269.666776  7.374779e+18
8     PA   3152.258973  7.879675e+18
1     CE   3792.428382  7.905146e+18
3     ES   3402.590285  8.082276e+18
10    PR   6444.995080  8.719665e+18
4     GO   3773.057348  8.826936e+18
9     PE   3677.701172  8.930456e+18
2     DF   3358.361732  9.057274e+18
12    RS   8486.335494  1.032015e+19
0     BA   6905.219209  1.655448e+19
11    RJ  14103.706753  2.199057e+19
6     MG  13721.566172  2.290257e+19


### 5. Consolidate & re-import

In [49]:
fct_volume_forecast = pd.concat(all_forecasts, ignore_index=True)
fct_volume_forecast.rename(columns={"ds": "forecast_date"}, inplace=True)

actual_part = state_df.rename(columns={"y": "value"})
actual_part["type"] = "actual"

predicted_part = forecast[["ds", "yhat"]].rename(columns={"yhat": "value"})
predicted_part["type"] = "predicted"

combined = pd.concat([actual_part, predicted_part], ignore_index=True)

fct_volume_forecast.to_sql(
    "fct_volume_forecast", engine, if_exists="replace", index=False
)

combined.to_sql(
    "fct_volume_forecast_actual_and_predicted", engine, if_exists="replace", index=False
)

224